"# Weakened-Transformer Crossover Search

This notebook demonstrates the **weaken-the-transformer crossover experiment**: comparing a TF-IDF + Logistic Regression baseline against deliberately-weakened transformer arms (DistilBERT truncated to 8/16 tokens, and `prajjwal1/bert-tiny`) on short-text sentiment classification, across a training-size grid, to look for a genuine finite empirical crossover point `n*` where the weak model overtakes the TF-IDF baseline.

Where a crossover is found, the notebook also runs the label-free **Good-Turing / Chao1 coverage-curve predictor**: it tries to predict the crossover point on one domain using only unlabeled text (no labels), calibrated on another domain.

This is a small-scale, fast-running demo of `method.py` from the original artifact, using a curated subset of the `rotten_tomatoes` domain data. Config parameters are reduced (fewer training sizes, seeds, epochs, and grid points) so the whole notebook runs in well under 10 minutes on CPU."]

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# transformers/torch are pre-installed on Colab but at versions that may lag; we rely on
# Colab's own torch+transformers install. loguru is NOT pre-installed on Colab.
_pip('loguru==0.7.3')

# Core packages pre-installed on Colab -- only install locally to match Colab's environment.
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0',
         'torch==2.9.0', 'transformers==5.0.0')

In [ ]:
from __future__ import annotations

import gc
import json
import random
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import matplotlib.pyplot as plt
from loguru import logger
from scipy import stats as sstats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

"## Load data

The original script reads `full_data_out.json`, a three-domain sentiment corpus (`tweet_eval`, `rotten_tomatoes`) produced by a separate dataset-generation artifact. For this demo we use `mini_demo_data.json`, a curated subset of the `rotten_tomatoes` domain (50 train rows nested at `n<=50`, 20 val, 20 test, 60 unlabeled-pool rows), loaded from GitHub with a local fallback."]

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-d8fc3d-vocabulary-saturation-predicts-the/main/round-2/experiment-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
DOMAINS = [g["dataset"] for g in data["datasets"]]
print("Domains:", DOMAINS)

"## Config

All tunable parameters from the original script, collected here. The original full-plan grid used `N_GRID = [50, 100, 200, 500, 1000]`, `SEEDS = [0, 1, 2]`, `n_epochs = 3`, `B_BOOTSTRAP = 100` bootstrap resamples per coverage-curve point, etc. -- infeasible to run at that scale in a short demo on CPU. Below we start at the **absolute minimum** (1 training size, 1 seed, 1 epoch, small bootstrap count) and only the demo-appropriate values are active; the true original values are commented alongside for reference."]

In [ ]:
N_GRID_FULL_PLAN = [50, 100, 200, 500, 1000, 1500, 2000]  # true original full plan grid (unused here)
N_GRID = [10, 25, 50]  # demo grid; original degraded grid was [50, 100, 200, 500, 1000]
SEEDS = [0]  # demo: 1 seed; original used [0, 1, 2]
MODEL_VARIANTS = ["tfidf_lr", "bert_tiny", "distilbert_trunc8", "distilbert_trunc16"]
WEAKENED_VARIANTS = ["bert_tiny", "distilbert_trunc8", "distilbert_trunc16"]

TFIDF_HPARAMS = {"ngram_range": (1, 2), "max_features": 20000, "min_df": 1}
LR_HPARAMS = {"C": 1.0, "penalty": "l2", "solver": "liblinear", "max_iter": 1000}
TRANSFORMER_HPARAMS = {
    "n_epochs": 1,  # demo: 1 epoch; original used 3
    "learning_rate": 3e-5,
    "batch_size": 8,  # demo: 8; original used 16
    "optimizer": "AdamW",
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
}
MODEL_CHECKPOINTS = {
    "bert_tiny": "prajjwal1/bert-tiny",
    "distilbert_trunc8": "distilbert-base-uncased",
    "distilbert_trunc16": "distilbert-base-uncased",
}
MAX_LENGTHS = {"bert_tiny": 32, "distilbert_trunc8": 8, "distilbert_trunc16": 16}

TIME_BUDGET_S = 480  # demo sweep wall-clock budget; original used 3.0*3600
PILOT_SIZES = [20, 30, 50]  # demo: small pilot sizes; original used [100, 200, 400]
COVERAGE_N_GRID = [10, 20, 40, 60]  # demo: matches small unlabeled pool; original used [50,100,200,400,800,1600,3200]
B_BOOTSTRAP = 20  # demo: 20 bootstrap resamples; original used 100
F2_INSTABILITY_THRESHOLD = 10
LAPLACE_SMOOTH = 0.5

DEGRADATION_LOG: list[dict[str, Any]] = [
    {
        "reason": "demo_notebook_scale_down",
        "detail": (
            "This notebook is a fast-running demo of the original method.py. Grid size, "
            "seed count, epoch count, and bootstrap count are all reduced from the original "
            "artifact run so the whole notebook completes in a few minutes on CPU."
        ),
        "n_grid_used": N_GRID,
        "n_grid_full_plan": N_GRID_FULL_PLAN,
        "seeds_used": SEEDS,
        "model_variants_used": MODEL_VARIANTS,
    }
]

"## Data helpers and model-fitting functions

These are copied directly from `method.py`: `load_domain_data` splits a domain's examples by `metadata_split`; `nested_train_subset` reconstructs the train pool for a given `n` using `metadata_train_min_n`; `fit_eval_tfidf_lr` and `fit_eval_transformer` fit and evaluate each model variant."]

In [ ]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    import torch

    torch.manual_seed(seed)


def load_domain_data(data: dict, domain: str) -> dict[str, list[dict]]:
    group = next(g for g in data["datasets"] if g["dataset"] == domain)
    out: dict[str, list[dict]] = {"val": [], "test": [], "train": [], "unlabeled_pool": []}
    for row in group["examples"]:
        out[row["metadata_split"]].append(row)
    return out


def nested_train_subset(train_rows: list[dict], n: int) -> list[dict]:
    subset = [r for r in train_rows if r.get("metadata_train_min_n") is not None and r["metadata_train_min_n"] <= n]
    return subset


def fit_eval_tfidf_lr(train_rows: list[dict], val_rows: list[dict], test_rows: list[dict], seed: int) -> dict:
    x_train = [r["input"] for r in train_rows]
    y_train = [int(r["output"]) for r in train_rows]
    vec = TfidfVectorizer(**TFIDF_HPARAMS)
    xt = vec.fit_transform(x_train)
    clf = LogisticRegression(random_state=seed, **LR_HPARAMS)
    clf.fit(xt, y_train)
    x_val = vec.transform([r["input"] for r in val_rows])
    x_test = vec.transform([r["input"] for r in test_rows])
    val_acc = clf.score(x_val, [int(r["output"]) for r in val_rows])
    test_acc = clf.score(x_test, [int(r["output"]) for r in test_rows])
    return {"val_acc": val_acc, "test_acc": test_acc}

In [ ]:
def fit_eval_transformer(model_variant: str, train_rows: list[dict], val_rows: list[dict], test_rows: list[dict], seed: int) -> dict:
    import torch
    from torch.utils.data import DataLoader, Dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

    torch.set_num_threads(4)
    set_all_seeds(seed)
    ckpt = MODEL_CHECKPOINTS[model_variant]
    max_len = MAX_LENGTHS[model_variant]
    tok = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2)
    device = torch.device("cpu")
    model.to(device)

    class TxtDS(Dataset):
        def __init__(self, rows):
            self.texts = [r["input"] for r in rows]
            self.labels = [int(r["output"]) for r in rows]

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, i):
            return self.texts[i], self.labels[i]

    def collate(batch):
        texts, labels = zip(*batch)
        enc = tok(list(texts), padding="max_length", truncation=True, max_length=max_len, return_tensors="pt")
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        assert enc["input_ids"].shape[1] <= max_len, "tokenized length exceeds configured max_length"
        return enc

    bs = TRANSFORMER_HPARAMS["batch_size"]
    train_loader = DataLoader(TxtDS(train_rows), batch_size=bs, shuffle=True, collate_fn=collate, generator=torch.Generator().manual_seed(seed))
    n_epochs = TRANSFORMER_HPARAMS["n_epochs"]
    total_steps = max(1, len(train_loader) * n_epochs)
    optimizer = torch.optim.AdamW(model.parameters(), lr=TRANSFORMER_HPARAMS["learning_rate"], weight_decay=TRANSFORMER_HPARAMS["weight_decay"])
    n_warmup = int(TRANSFORMER_HPARAMS["warmup_ratio"] * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=n_warmup, num_training_steps=total_steps)

    model.train()
    nan_loss_detected = False
    for _epoch in range(n_epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            if torch.isnan(loss):
                nan_loss_detected = True
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

    def evaluate(rows: list[dict]) -> float:
        model.eval()
        correct = 0
        loader = DataLoader(TxtDS(rows), batch_size=32, shuffle=False, collate_fn=collate)
        with torch.no_grad():
            for batch in loader:
                labels = batch.pop("labels")
                batch = {k: v.to(device) for k, v in batch.items()}
                logits = model(**batch).logits
                preds = torch.argmax(logits, dim=-1)
                correct += (preds.cpu() == labels).sum().item()
        return correct / len(rows)

    val_acc = evaluate(val_rows)
    test_acc = evaluate(test_rows)
    del model, optimizer, scheduler, train_loader
    gc.collect()
    return {"val_acc": val_acc, "test_acc": test_acc, "nan_loss_detected": nan_loss_detected}

"## Main sweep

Fits every (domain, model_variant, n, seed) cell and records test accuracy, with a wall-clock guard that degrades gracefully if the time budget runs out. No checkpoint file is used in the notebook (single short run), unlike the original script which resumes from `method_out_partial.jsonl`."]

In [ ]:
def run_main_sweep(all_domain_data: dict[str, dict]) -> tuple[list[dict], list[dict]]:
    results: list[dict] = []
    degradations: list[dict] = list(DEGRADATION_LOG)
    start_time = time.time()

    def remaining_time() -> float:
        return TIME_BUDGET_S - (time.time() - start_time)

    cells = [(d, mv, n, s) for d in DOMAINS for mv in MODEL_VARIANTS for n in N_GRID for s in SEEDS]
    n_cells = len(cells)
    logger.info(f"Main sweep: {n_cells} cells over domains={DOMAINS}, variants={MODEL_VARIANTS}, n_grid={N_GRID}, seeds={SEEDS}")

    done_count = 0
    for domain, model_variant, n, seed in cells:
        done_count += 1
        if remaining_time() < 15:
            logger.warning(f"Wall-clock guard tripped before cell {done_count}/{n_cells}; degrading rest.")
            degradations.append(
                {
                    "reason": "wall_clock_guard",
                    "domain": domain,
                    "model_variant": model_variant,
                    "n": n,
                    "seed": seed,
                    "elapsed_s": time.time() - start_time,
                }
            )
            continue
        pool = all_domain_data[domain]
        train_subset = nested_train_subset(pool["train"], n)
        val_rows, test_rows = pool["val"], pool["test"]
        t0, cpu0 = time.time(), time.process_time()
        try:
            if model_variant == "tfidf_lr":
                metrics = fit_eval_tfidf_lr(train_subset, val_rows, test_rows, seed)
            else:
                metrics = fit_eval_transformer(model_variant, train_subset, val_rows, test_rows, seed)
        except Exception:
            logger.exception(f"Cell failed: {domain}/{model_variant}/n={n}/seed={seed}")
            degradations.append({"reason": "exception", "domain": domain, "model_variant": model_variant, "n": n, "seed": seed})
            continue
        wall, cpu = time.time() - t0, time.process_time() - cpu0
        row = {
            "domain": domain,
            "model_variant": model_variant,
            "n": n,
            "seed": seed,
            "n_train_actual": len(train_subset),
            "val_acc": metrics["val_acc"],
            "test_acc": metrics["test_acc"],
            "wall_clock_s": wall,
            "cpu_time_s": cpu,
            "degraded": False,
        }
        results.append(row)
        logger.info(
            f"[{done_count}/{n_cells}] {domain}/{model_variant}/n={n}/seed={seed}: "
            f"test_acc={metrics['test_acc']:.4f} wall={wall:.1f}s (remaining budget {remaining_time():.0f}s)"
        )
        gc.collect()

    return results, degradations

In [ ]:
all_domain_data = {d: load_domain_data(data, d) for d in DOMAINS}
for d in DOMAINS:
    logger.info(f"{d}: train={len(all_domain_data[d]['train'])} val={len(all_domain_data[d]['val'])} test={len(all_domain_data[d]['test'])} pool={len(all_domain_data[d]['unlabeled_pool'])}")

logger.info("=== MAIN SWEEP ===")
sweep_start = time.time()
results, degradations = run_main_sweep(all_domain_data)
sweep_wall_s = time.time() - sweep_start
logger.info(f"Main sweep done: {len(results)} cells completed, {len(degradations) - 1} degradation events, wall={sweep_wall_s:.1f}s")

"## Crossover detection

For each (domain, weakened-model) pair, compute the gap curve (weak-model minus TF-IDF test accuracy) and its pooled cross-seed standard error, and find the first grid `n` where the gap exceeds the pooled SE -- the empirical crossover point `n*`."]

In [ ]:
def compute_gap_and_pooled_se(results: list[dict], domain: str, weak_variant: str, n_grid: list[int], seeds: list[int]) -> list[dict]:
    curve = []
    for n in n_grid:
        weak_accs = [r["test_acc"] for r in results if r["domain"] == domain and r["model_variant"] == weak_variant and r["n"] == n]
        base_accs = [r["test_acc"] for r in results if r["domain"] == domain and r["model_variant"] == "tfidf_lr" and r["n"] == n]
        if not weak_accs or not base_accs:
            curve.append({"n": n, "gap": None, "pooled_se": None, "weak_mean": None, "base_mean": None, "n_weak_seeds": len(weak_accs), "n_base_seeds": len(base_accs)})
            continue
        weak_mean, base_mean = float(np.mean(weak_accs)), float(np.mean(base_accs))
        weak_se = float(np.std(weak_accs, ddof=1) / np.sqrt(len(weak_accs))) if len(weak_accs) > 1 else 0.0
        base_se = float(np.std(base_accs, ddof=1) / np.sqrt(len(base_accs))) if len(base_accs) > 1 else 0.0
        pooled_se = float(np.sqrt(weak_se**2 + base_se**2))
        curve.append(
            {
                "n": n,
                "gap": weak_mean - base_mean,
                "pooled_se": pooled_se,
                "weak_mean": weak_mean,
                "base_mean": base_mean,
                "n_weak_seeds": len(weak_accs),
                "n_base_seeds": len(base_accs),
            }
        )
    return curve


def first_n_where_gap_exceeds_se(gap_curve: list[dict]) -> int | None:
    for pt in gap_curve:
        if pt["gap"] is not None and pt["pooled_se"] is not None and pt["gap"] > pt["pooled_se"]:
            return pt["n"]
    return None

"## Label-free coverage predictor

This is the label-free Good-Turing/Chao1 coverage-curve predictor protocol: build a discriminative vocabulary from a small labeled pilot (MI contingency-table selection with a permutation null, plus a frequency-delta variant), then compute a bootstrap coverage curve over the domain's *unlabeled* pool. `fit_tau`/`predict_crossover` calibrate a missing-mass threshold `tau` on one (domain, variant) pair's true `n*` and try to predict another pair's `n*` from coverage alone."]

In [ ]:
def tokenize_simple(text: str) -> list[str]:
    return [t.lower() for t in text.split() if t.strip()]


def build_ngram_vocab(texts: list[str], ngram_range=(1, 2)) -> list[list[str]]:
    docs_ngrams = []
    for t in texts:
        toks = tokenize_simple(t)
        grams = list(toks)
        for i in range(len(toks) - 1):
            grams.append(toks[i] + "_" + toks[i + 1])
        docs_ngrams.append(sorted(set(grams)))
    return docs_ngrams


def mi_contingency(present_pos: int, present_neg: int, absent_pos: int, absent_neg: int, smooth: float = LAPLACE_SMOOTH) -> float:
    n11, n10, n01, n00 = present_pos + smooth, present_neg + smooth, absent_pos + smooth, absent_neg + smooth
    n = n11 + n10 + n01 + n00
    mi = 0.0
    for n_xy, n_x, n_y in [
        (n11, n11 + n10, n11 + n01),
        (n10, n11 + n10, n10 + n00),
        (n01, n01 + n00, n11 + n01),
        (n00, n01 + n00, n10 + n00),
    ]:
        p_xy, p_x, p_y = n_xy / n, n_x / n, n_y / n
        if p_xy > 0 and p_x > 0 and p_y > 0:
            mi += p_xy * np.log2(p_xy / (p_x * p_y))
    return float(mi)


def build_discriminative_vocab(pilot_rows: list[dict], top_k: int = 200, min_df: int = 2, n_perm: int = 200, seed: int = 0) -> dict:
    texts = [r["input"] for r in pilot_rows]
    labels = np.array([int(r["output"]) for r in pilot_rows])
    docs_ngrams = build_ngram_vocab(texts)
    term_doc_count: dict[str, int] = {}
    for grams in docs_ngrams:
        for g in grams:
            term_doc_count[g] = term_doc_count.get(g, 0) + 1
    candidate_terms = [t for t, c in term_doc_count.items() if c >= min_df]
    n_pos, n_neg = int(labels.sum()), int((1 - labels).sum())

    def compute_mi_for_terms(terms: list[str], lbls: np.ndarray) -> dict[str, float]:
        mi_scores = {}
        for t in terms:
            present = np.array([1 if t in grams else 0 for grams in docs_ngrams])
            pp = int(((present == 1) & (lbls == 1)).sum())
            pn = int(((present == 1) & (lbls == 0)).sum())
            ap = int(((present == 0) & (lbls == 1)).sum())
            an = int(((present == 0) & (lbls == 0)).sum())
            mi_scores[t] = mi_contingency(pp, pn, ap, an)
        return mi_scores

    mi_scores = compute_mi_for_terms(candidate_terms, labels)
    freq_scores: dict[str, float] = {}
    for t in candidate_terms:
        present = np.array([1 if t in grams else 0 for grams in docs_ngrams])
        p_pos = present[labels == 1].mean() if n_pos > 0 else 0.0
        p_neg = present[labels == 0].mean() if n_neg > 0 else 0.0
        freq_scores[t] = abs(float(p_pos) - float(p_neg))

    rng = np.random.RandomState(seed)
    max_mi_null = []
    n_perm_terms = candidate_terms if len(candidate_terms) <= 500 else list(rng.choice(candidate_terms, 500, replace=False))
    for _ in range(n_perm):
        shuffled = rng.permutation(labels)
        perm_scores = compute_mi_for_terms(n_perm_terms, shuffled)
        max_mi_null.append(max(perm_scores.values()) if perm_scores else 0.0)
    mi_cutoff = float(np.percentile(max_mi_null, 95)) if max_mi_null else 0.0

    mi_selected = sorted([t for t, s in mi_scores.items() if s > mi_cutoff], key=lambda t: -mi_scores[t])[:top_k]
    if not mi_selected:
        mi_selected = sorted(candidate_terms, key=lambda t: -mi_scores[t])[:top_k]
    freq_selected = sorted(candidate_terms, key=lambda t: -freq_scores[t])[:top_k]

    jaccard = len(set(mi_selected) & set(freq_selected)) / max(1, len(set(mi_selected) | set(freq_selected)))
    return {
        "vocab_mi": mi_selected,
        "vocab_freq": freq_selected,
        "mi_cutoff": mi_cutoff,
        "jaccard_mi_vs_freq": jaccard,
        "pilot_size": len(pilot_rows),
    }

In [ ]:
def good_turing_coverage_curve(pool_texts: list[str], vocab: list[str], n_grid: list[int], b: int = B_BOOTSTRAP, seed: int = 0) -> list[dict]:
    rng = np.random.RandomState(seed)
    pool_docs_terms = []
    vocab_set = set(vocab)
    for t in pool_texts:
        toks = tokenize_simple(t)
        grams = set(toks)
        for i in range(len(toks) - 1):
            grams.add(toks[i] + "_" + toks[i + 1])
        pool_docs_terms.append(grams & vocab_set)
    n_pool = len(pool_docs_terms)
    curve = []
    for n in n_grid:
        n_eff = min(n, n_pool)
        c_vals, chao1_vals, f2_lt10_flags = [], [], []
        for _ in range(b):
            idx = rng.choice(n_pool, size=n_eff, replace=False)
            counts: dict[str, int] = {}
            for i in idx:
                for term in pool_docs_terms[i]:
                    counts[term] = counts.get(term, 0) + 1
            freq_of_freq: dict[int, int] = {}
            for c in counts.values():
                freq_of_freq[c] = freq_of_freq.get(c, 0) + 1
            f1 = freq_of_freq.get(1, 0)
            f2 = freq_of_freq.get(2, 0)
            n_total = sum(counts.values())
            c_hat = 1 - (f1 / n_total if n_total > 0 else 1.0)
            s_obs = len(counts)
            chao1 = s_obs + (f1 * (f1 - 1) / (2 * (f2 + 1)))
            c_vals.append(c_hat)
            chao1_vals.append(chao1)
            f2_lt10_flags.append(f2 < F2_INSTABILITY_THRESHOLD)
        unstable_frac = float(np.mean(f2_lt10_flags))
        curve.append(
            {
                "n": n,
                "n_effective": n_eff,
                "coverage_median": float(np.median(c_vals)),
                "coverage_ci_lo": float(np.percentile(c_vals, 2.5)),
                "coverage_ci_hi": float(np.percentile(c_vals, 97.5)),
                "chao1_median": float(np.median(chao1_vals)),
                "unstable_fraction": unstable_frac,
                "unstable": unstable_frac > 0.5,
            }
        )
    return curve


def fit_tau(coverage_curve: list[dict], n_star: int) -> float | None:
    pt = next((c for c in coverage_curve if c["n"] == n_star), None)
    if pt is None:
        closest = min(coverage_curve, key=lambda c: abs(c["n"] - n_star))
        pt = closest
    return 1.0 - pt["coverage_median"]


def predict_crossover(coverage_curve: list[dict], tau: float) -> int | None:
    for pt in sorted(coverage_curve, key=lambda c: c["n"]):
        if (1.0 - pt["coverage_median"]) < tau:
            return pt["n"]
    return None

"## Run crossover detection + coverage predictor

This mirrors the body of `main()` in `method.py`: detect crossover pairs, then (if any found) build vocab + coverage curves for each and run every directed calibrate/predict test between pairs."]

In [ ]:
logger.info("=== CROSSOVER DETECTION ===")
all_pairs = []
for domain in DOMAINS:
    for mv in WEAKENED_VARIANTS:
        gap_curve = compute_gap_and_pooled_se(results, domain, mv, N_GRID, SEEDS)
        n_star = first_n_where_gap_exceeds_se(gap_curve)
        all_pairs.append({"domain": domain, "model_variant": mv, "n_star": n_star, "gap_curve": gap_curve})
        logger.info(f"{domain}/{mv}: n_star={n_star}")

crossover_pairs = [p for p in all_pairs if p["n_star"] is not None]

predictor_results: list[dict] = []
negative_result = None
if crossover_pairs:
    logger.info(f"=== COVERAGE PREDICTOR ({len(crossover_pairs)} crossover pair(s) found) ===")
    for pair in crossover_pairs:
        domain = pair["domain"]
        pool_texts = [r["input"] for r in all_domain_data[domain]["unlabeled_pool"]]
        pilot_size = min(PILOT_SIZES[-1], len(all_domain_data[domain]["train"]))
        pilot_rows = all_domain_data[domain]["train"][:pilot_size]
        vocab_info = build_discriminative_vocab(pilot_rows)
        pair["vocab_info"] = vocab_info
        pair["coverage_curve_mi"] = good_turing_coverage_curve(pool_texts, vocab_info["vocab_mi"], COVERAGE_N_GRID)
        pair["coverage_curve_freq"] = good_turing_coverage_curve(pool_texts, vocab_info["vocab_freq"], COVERAGE_N_GRID)
        logger.info(f"{domain}/{pair['model_variant']}: vocab built (jaccard mi-vs-freq={vocab_info['jaccard_mi_vs_freq']:.3f})")

    for calib in crossover_pairs:
        for predict in crossover_pairs:
            if calib is predict:
                continue
            tau = fit_tau(calib["coverage_curve_mi"], calib["n_star"])
            n_hat = predict_crossover(predict["coverage_curve_mi"], tau)
            within_2x = n_hat is not None and (predict["n_star"] / 2 <= n_hat <= predict["n_star"] * 2)
            shared_ns = [pt["n"] for pt in predict["coverage_curve_mi"] if pt["n"] in N_GRID]
            cov_vals = [1 - pt["coverage_median"] for pt in predict["coverage_curve_mi"] if pt["n"] in N_GRID]
            gap_vals = [gp["gap"] for gp in predict["gap_curve"] if gp["n"] in shared_ns and gp["gap"] is not None]
            rho, p_val = (None, None)
            if len(cov_vals) >= 3 and len(gap_vals) == len(cov_vals):
                rho, p_val = sstats.spearmanr(cov_vals, gap_vals)
                rho, p_val = float(rho), float(p_val)
            predictor_results.append(
                {
                    "calibrate_on": f"{calib['domain']}/{calib['model_variant']}",
                    "predict_on": f"{predict['domain']}/{predict['model_variant']}",
                    "tau": tau,
                    "n_hat_star": n_hat,
                    "true_n_star": predict["n_star"],
                    "within_2x": within_2x,
                    "spearman_rho": rho,
                    "spearman_p": p_val,
                }
            )
else:
    negative_result = {
        "claim": (
            "No (domain, weakened-model) pair among "
            f"{DOMAINS} x {WEAKENED_VARIANTS} produced a finite empirical "
            f"crossover n* within the grid {N_GRID}."
        ),
        "interpretation": (
            "The coverage predictor remains untested against a real target on this demo-scale grid."
        ),
    }
    logger.warning("No crossover pairs found; recording explicit negative result.")

any_crossover_found = len(crossover_pairs) > 0
predictor_validated = any(pr["within_2x"] for pr in predictor_results) if predictor_results else None
if not any_crossover_found:
    verdict = "NO_CROSSOVER_FOUND: no finite crossover on this demo grid; predictor remains untested."
elif predictor_validated is True:
    verdict = f"CROSSOVER_FOUND_AND_PREDICTOR_VALIDATED: {len(crossover_pairs)} crossover pair(s) found; >=1 calibrate/predict direction landed within the 2x band."
elif predictor_validated is False:
    verdict = f"CROSSOVER_FOUND_PREDICTOR_REFUTED: {len(crossover_pairs)} crossover pair(s) found; no calibrate/predict direction landed within the 2x band."
else:
    verdict = f"CROSSOVER_FOUND_PREDICTOR_UNTESTED: {len(crossover_pairs)} crossover pair(s) found but <2 pairs available so no cross-pair calibrate/predict test could be run."

logger.info(f"TOP-LINE VERDICT: {verdict}")

"## Results

Print the top-line verdict and per-cell accuracy table, then plot test accuracy vs. training-set size for each model variant (the crossover plot) and the gap curves against pooled standard error."]

In [ ]:
print("TOP-LINE VERDICT:", verdict)
print()
print(f"{'domain':<16}{'model_variant':<20}{'n':>6}{'seed':>6}{'test_acc':>10}")
for r in sorted(results, key=lambda r: (r["domain"], r["model_variant"], r["n"], r["seed"])):
    print(f"{r['domain']:<16}{r['model_variant']:<20}{r['n']:>6}{r['seed']:>6}{r['test_acc']:>10.4f}")

print()
print("Crossover pairs (domain/weakened-variant -> n*):")
for p in all_pairs:
    print(f"  {p['domain']}/{p['model_variant']}: n_star={p['n_star']}")

if predictor_results:
    print()
    print("Calibrate/predict directions:")
    for pr in predictor_results:
        print(f"  calibrate={pr['calibrate_on']:<28} predict={pr['predict_on']:<28} "
              f"n_hat*={pr['n_hat_star']} true_n*={pr['true_n_star']} within_2x={pr['within_2x']}")

# --- Plot: test accuracy vs training size, one line per model variant ---
fig, axes = plt.subplots(1, len(DOMAINS), figsize=(6 * len(DOMAINS), 4.5), squeeze=False)
for ax, domain in zip(axes[0], DOMAINS):
    for mv in MODEL_VARIANTS:
        ns, accs = [], []
        for n in N_GRID:
            cell_accs = [r["test_acc"] for r in results if r["domain"] == domain and r["model_variant"] == mv and r["n"] == n]
            if cell_accs:
                ns.append(n)
                accs.append(float(np.mean(cell_accs)))
        if ns:
            ax.plot(ns, accs, marker="o", label=mv)
    ax.set_title(domain)
    ax.set_xlabel("training set size n")
    ax.set_ylabel("test accuracy")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.suptitle("Weaken-the-transformer crossover search (demo scale)")
plt.tight_layout()
plt.show()